# Activity: MCP Tool Discovery and Invocation
In this activity, you drive an MCP client against this module's ChemE tools server: connect and initialize the session, discover the tools and their input schemas, and play the agent by turning natural-language requests into schema-conformant tool calls.

__Why are we looking at this?__ In a production deployment, the host sends tool descriptors to a language model, and the model selects tools and constructs arguments. There is no model here: you make those decisions yourself, which shows exactly what information the protocol gives an agent and what the agent must do with it.

> __Learning Objectives__
>
> By the end of this activity, you will be able to:
> * __Run the handshake:__ Connect to the server, run the initialization handshake, and read the server identity from `serverInfo`.
> * __Construct schema-conformant arguments:__ Select the tool that answers a natural-language request and build its arguments dictionary from the tool's `inputSchema`.
> * __Interpret both failure modes:__ Distinguish a tool execution error (a `result` with `isError` set to `true`) from a protocol error (a JSON-RPC error object), and explain why the server answers through different channels.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file, which activates the local environment, loads the required packages, and includes the MCP client and server code in `src/`.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-4`


### Constants
We build the command that launches the server subprocess: the Julia executable, this module's project, and the `server.jl` entry point. This is the same launch-command shape a host application stores in its MCP configuration file.

In [2]:
server_command = `$(joinpath(Sys.BINDIR, "julia")) --project=$(_ROOT) $(_PATH_TO_SERVER)`;

## Task 1: Connect and Discover
Connect to the server and run the initialization handshake. The `connect(...)` call launches the subprocess; `initialize!(...)` sends the `initialize` request and the `notifications/initialized` notification. With `verbose = true`, every wire message is echoed: `→` outgoing, `←` incoming. The `@assert` checks that the server identifies itself as this module's server, and the last line displays the `serverInfo` block.

In [3]:
connection = connect(server_command, verbose = true);
response_initialize = initialize!(connection);
@assert response_initialize["result"]["serverInfo"]["name"] == "cheme-142-m4-cheme-tools"
response_initialize["result"]["serverInfo"]

→ {"method":"initialize","id":1,"params":{"clientInfo":{"name":"cheme-142-notebook-client","version":"1.0.0"},"protocolVersion":"2025-06-18","capabilities":{}},"jsonrpc":"2.0"}

  Activating project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-4`


← {"id":1,"jsonrpc":"2.0","result":{"protocolVersion":"2025-06-18","capabilities":{"tools":{}},"serverInfo":{"name":"cheme-142-m4-cheme-tools","version":"1.0.0"}}}
→ {"method":"notifications/initialized","jsonrpc":"2.0"}


Dict{String, Any} with 2 entries:
  "name"    => "cheme-142-m4-cheme-tools"
  "version" => "1.0.0"

### Discover the Tools
Ask the server what it can do. The `tools/list` request returns one descriptor per tool, listed alphabetically by name. As you read the output, answer for yourself: which tool answers a saturation-pressure question, and what does its schema require? The second cell prints that tool's `inputSchema` in full.

In [4]:
tools = listtools(connection)["result"]["tools"];
[t["name"] for t in tools]

→ {"method":"tools/list","id":2,"params":{},"jsonrpc":"2.0"}
← {"id":2,"jsonrpc":"2.0","result":{"tools":[{"name":"antoine_vapor_pressure","inputSchema":{"properties":{"T":{"type":"number","description":"Temperature in K, inside the species' valid range"},"species":{"type":"string","description":"Species name, e.g. water, acetone, ethanol, benzene"}},"required":["species","T"],"type":"object"},"description":"Saturation pressure (bar) of a named species at temperature T (K) from the Antoine equation."},{"name":"ideal_gas_solve","inputSchema":{"properties":{"T":{"type":"number","description":"Temperature in K"},"P":{"type":"number","description":"Pressure in Pa"},"V":{"type":"number","description":"Volume in m^3"},"n":{"type":"number","description":"Amount in mol"}},"required":[],"type":"object"},"description":"Solve the ideal gas law PV = nRT for the one variable not provided (SI units: Pa, m^3, mol, K)."},{"name":"molecular_weight","inputSchema":{"properties":{"formula":{"type":"string

3-element Vector{String}:
 "antoine_vapor_pressure"
 "ideal_gas_solve"
 "molecular_weight"

In [5]:
JSON.print(tools[1]["inputSchema"], 2)

{
  "properties": {
    "T": {
      "type": "number",
      "description": "Temperature in K, inside the species' valid range"
    },
    "species": {
      "type": "string",
      "description": "Species name, e.g. water, acetone, ethanol, benzene"
    }
  },
  "required": [
    "species",
    "T"
  ],
  "type": "object"
}


___

## Task 2: Play the Agent
Here is a natural-language request:

> *"What is the saturation pressure of acetone at 320 K, in bar?"*

An agent facing this request makes three decisions:

1. __Which tool?__ The request asks for a saturation pressure, and the description of `antoine_vapor_pressure` promises exactly this quantity.
2. __Which arguments?__ The tool's `inputSchema` requires a string `species` and a number `T` in kelvin, so the arguments dictionary is `species = "acetone"`, `T = 320.0`.
3. __What does the result mean?__ The result's text payload carries the saturation pressure `Psat` in bar, with the species and temperature echoed back.

The cell below makes the call and checks the answer.

In [6]:
arguments_psat = Dict("species" => "acetone", "T" => 320.0); # from the inputSchema: species (string), T (number, K)
response_psat = calltool(connection, "antoine_vapor_pressure", arguments_psat);
result_psat = JSON.parse(response_psat["result"]["content"][1]["text"]);
@assert isapprox(result_psat["Psat"], 0.7261; rtol = 1e-2)
result_psat

→ {"method":"tools/call","id":3,"params":{"name":"antoine_vapor_pressure","arguments":{"T":320.0,"species":"acetone"}},"jsonrpc":"2.0"}

← {"id":3,"jsonrpc":"2.0","result":{"content":[{"text":"{\"units\":\"bar\",\"T\":320.0,\"Psat\":0.7261,\"species\":\"acetone\"}","type":"text"}],"isError":false}}


Dict{String, Any} with 4 entries:
  "units"   => "bar"
  "T"       => 320.0
  "Psat"    => 0.7261
  "species" => "acetone"

__Modify and re-run:__ Change the species to `"ethanol"` at the same temperature and re-run the cell (the `@assert` checks the acetone answer, so comment it out or update its value first). Which entry of the result changed, and why is the pressure lower than acetone's at the same temperature?
___

### A Second Request
Another request:

> *"How many moles of an ideal gas occupy 10 L at 1 bar and 298.15 K?"*

The tool is `ideal_gas_solve`, which solves $PV = nRT$ for whichever of $P$, $V$, $n$, $T$ is not provided. Its schema demands unit discipline: the descriptions say pressure in Pa and volume in m³, so 1 bar becomes $1.0\times10^{5}$ Pa and 10 L becomes $0.010$ m³. We omit `n`, so the server solves for it.

In [7]:
arguments_n = Dict("P" => 1.0e5, "V" => 0.010, "T" => 298.15);
response_n = calltool(connection, "ideal_gas_solve", arguments_n);
result_n = JSON.parse(response_n["result"]["content"][1]["text"]);
@assert result_n["variable"] == "n"
@assert isapprox(result_n["value"], 0.40342; rtol = 1e-3)
result_n

→ {"method":"tools/call","id":4,"params":{"name":"ideal_gas_solve","arguments":{"T":298.15,"P":100000.0,"V":0.01}},"jsonrpc":"2.0"}


← {"id":4,"jsonrpc":"2.0","result":{"content":[{"text":"{\"units\":\"mol\",\"value\":0.403418,\"variable\":\"n\"}","type":"text"}],"isError":false}}


Dict{String, Any} with 3 entries:
  "units"    => "mol"
  "value"    => 0.403418
  "variable" => "n"

__Modify and re-run:__ Pass all four of `P`, `V`, `n`, and `T` (add `"n" => 1.0` to the dictionary) and re-run the cell, commenting out the parse and `@assert` lines and inspecting `response_n` instead. What comes back, and which of the two failure modes is it? The next task names them.
___

___

## Task 3: Read the Failure Modes
Both failure modes appear when we push on the server. First, a valid request whose tool runs and fails: water at 200.0 K is below the Antoine table's valid range, so the tool throws and the server returns a `result` with `isError` set to `true` and the failure text in `content`. Second, a request that names a tool the server does not have: `fugacity` is unknown, so the server returns a JSON-RPC error object with code `-32602` and no `result`.

In [8]:
response_range = calltool(connection, "antoine_vapor_pressure", Dict("species" => "water", "T" => 200.0));
@assert response_range["result"]["isError"] == true
response_range["result"]["content"][1]["text"]

→ {"method":"tools/call","id":5,"params":{"name":"antoine_vapor_pressure","arguments":{"T":200.0,"species":"water"}},"jsonrpc":"2.0"}
← {"id":5,"jsonrpc":"2.0","result":{"content":[{"text":"ArgumentError: T = 200.0 K is outside the valid range [255.9, 373.0] K for water","type":"text"}],"isError":true}}


"ArgumentError: T = 200.0 K is outside the valid range [255.9, 373.0] K for water"

In [9]:
response_unknown = calltool(connection, "fugacity", Dict("species" => "water"));
@assert haskey(response_unknown, "error")
response_unknown["error"]

→ {"method":"tools/call","id":6,"params":{"name":"fugacity","arguments":{"species":"water"}},"jsonrpc":"2.0"}
← {"error":{"message":"Unknown tool: fugacity","code":-32602},"id":6,"jsonrpc":"2.0"}


Dict{String, Any} with 2 entries:
  "message" => "Unknown tool: fugacity"
  "code"    => -32602

__Explain:__ In one paragraph, state in your own words why the server answered these two calls through different channels. What makes the out-of-range temperature a tool execution error but the unknown tool a protocol error, and why would a host treat them differently?
___

## Shut Down
Close the connection. This closes the server's standard input, and the server's dispatch loop exits when its input reaches EOF.

In [10]:
close(connection);

## Summary
This activity drove a complete MCP session by hand: handshake, discovery, schema-conformant tool calls, and both failure modes.

> __Key Takeaways:__
>
> * **Handshake before use:** Every session starts with the `initialize` request and the `notifications/initialized` notification; only then does the client issue `tools/list` and `tools/call` requests.
> * **Schema-driven argument construction:** A tool's `inputSchema` is the contract for building arguments: it names each parameter and its type, marks which are required, and states units in the descriptions, and the caller must conform to it.
> * **Two failure channels:** A valid call whose tool fails returns a `result` with `isError` set to `true`, while an invalid request returns a JSON-RPC error object; a host reports the first back to the model and treats the second as a malformed request.

The graded activity moves you to the other side of the protocol: implementing and registering a server tool of your own.
___